In [41]:
import torch
from torch_geometric.nn.kge import DistMult
from torch_geometric.data import Dataset, HeteroData
from torch.utils.data import TensorDataset, DataLoader, random_split
import pickle
import pandas as pd
from tqdm import tqdm

In [42]:
df = pd.read_csv("data/100k.csv")

df_rev = df.rename(
    columns={'protein_1': 'protein_2', 'protein_2': 'protein_1'}
)

df = (
    pd.concat([df, df_rev], ignore_index=True)
      .drop_duplicates()
      .reset_index(drop=True))

uniq = pd.Index(pd.concat([df["protein_1"], df["protein_2"]]).unique())
id2idx = {int(x): i for i, x in enumerate(uniq)}

df["h_idx"] = df["protein_1"].map(id2idx).astype("int64")
df["t_idx"] = df["protein_2"].map(id2idx).astype("int64")
df

,protein_1,relation,protein_2,h_idx,t_idx
0,1127498,0,902,0,16
1,1126212,0,902,1,16
2,7156,0,902,2,16
3,1138775,0,902,3,16
4,1174,0,902,4,16
...,...,...,...,...,...
193362,1856,0,1158100,661,1069
193363,1856,0,2443,661,3991
193364,1856,0,3037,661,798
193365,1856,0,1142843,661,1140


In [44]:
def df_to_heterodata(
    df: pd.DataFrame,
    src_id_col="id_1",
    src_type_col="type_1",
    rel_col="interaction",
    dst_id_col="id_2",
    dst_type_col="type_2",
    make_undirected_for: set[str] | None = None,  # напр. {"interacts_with"}
):
    """
    df columns: id_1, type_1, interaction, id_2, type_2
    -> HeteroData with:
       data[node_type].num_nodes
       data[(src_type, rel, dst_type)].edge_index
    """
    make_undirected_for = make_undirected_for or set()

    data = HeteroData()

    # 1) Глобальные маппинги: (node_type -> {raw_id -> local_idx})
    id2idx: dict[str, dict[int, int]] = {}

    def get_local_idx(node_type: str, raw_id: int) -> int:
        m = id2idx.setdefault(node_type, {})
        if raw_id not in m:
            m[raw_id] = len(m)
        return m[raw_id]

    # 2) Собираем ребра по типам отношений
    edge_buckets: dict[tuple[str, str, str], list[tuple[int, int]]] = {}

    for r in df.itertuples(index=False):
        s_type = getattr(r, src_type_col)
        d_type = getattr(r, dst_type_col)
        rel    = getattr(r, rel_col)

        s_id = int(getattr(r, src_id_col))
        d_id = int(getattr(r, dst_id_col))

        s = get_local_idx(s_type, s_id)
        d = get_local_idx(d_type, d_id)

        key = (s_type, rel, d_type)
        edge_buckets.setdefault(key, []).append((s, d))

        # если связь симметрична (типа interacts_with) — добавим обратное ребро
        if rel in make_undirected_for:
            key_rev = (d_type, rel, s_type)
            edge_buckets.setdefault(key_rev, []).append((d, s))

    # 3) Заполняем узлы
    for ntype, mapping in id2idx.items():
        data[ntype].num_nodes = len(mapping)
        # Если есть фичи, то тут нужно поставить data[ntype].x = ...

    # 4) Заполняем ребра (edge_index)
    for (s_type, rel, d_type), pairs in edge_buckets.items():
        if len(pairs) == 0:
            continue
        edge_index = torch.tensor(pairs, dtype=torch.long).t().contiguous()  # [2, E]
        data[(s_type, rel, d_type)].edge_index = edge_index

    # (опционально) сохранить маппинги, чтобы потом возвращаться к исходным id
    data._raw_id2idx = id2idx

    return data



data = df_to_heterodata(df, make_undirected_for={"interacts_with", "has_similarity"})
# print(data)
# print(data.metadata())  # (node_types, edge_types)


In [43]:
head = torch.tensor(df["h_idx"].values, dtype=torch.int64)
tail = torch.tensor(df["t_idx"].values, dtype=torch.int64)
rel  = torch.tensor(df["relation"].values, dtype=torch.int64)


In [47]:
with open("data/dict.pkl", "rb") as f:
    emb_dict = pickle.load(f)

E_fixed = torch.stack([emb_dict[int(raw_id)] for raw_id in uniq], dim=0)
E_fixed


tensor([[-0.1620, -0.1617,  0.1418,  ...,  0.3633, -0.0133, -0.0952],
        [-0.0116, -0.0324,  0.1359,  ...,  0.2133, -0.0692, -0.0008],
        [-0.0731, -0.0250,  0.0899,  ...,  0.0727,  0.1244, -0.0475],
        ...,
        [-0.1131, -0.0875,  0.2915,  ...,  0.4812, -0.1141, -0.0569],
        [-0.0466, -0.1847,  0.1268,  ...,  0.0218,  0.1393, -0.0787],
        [-0.0304, -0.1642,  0.0984,  ...,  0.2135, -0.1006,  0.0475]],
       dtype=torch.float16)

In [45]:
# ---- входные данные ----
# E_fixed: [num_nodes, hidden_channels] (твои готовые эмбеддинги сущностей)
# head, rel, tail: LongTensor [B] с индексами триплетов
num_nodes = E_fixed.size(0)
hidden_channels = E_fixed.size(1)
num_relations = 1  # или задай явно

model = DistMult(
    num_nodes=num_nodes,
    num_relations=num_relations,
    hidden_channels=hidden_channels,
    margin=1.0,
)

In [46]:
# 1) загрузить фиксированные entity-эмбеддинги в node_emb
with torch.no_grad():
    model.node_emb.weight.copy_(E_fixed)

# 2) заморозить entity-эмбеддинги
model.node_emb.weight.requires_grad_(False)

# 3) оптимизатор только для relation-эмбеддингов
opt = torch.optim.Adam([model.rel_emb.weight], lr=1e-3)

dataset = TensorDataset(head, rel, tail)  # все 1D LongTensor одинаковой длины
train_set, test_set = random_split(dataset, [0.8, 0.2])

train_loader = DataLoader(train_set, batch_size=4096, shuffle=True)


# ---- train step (учится только rel_emb) ----
model.train()
for epoch in tqdm(range(20), leave=True):
    for head, rel, tail in tqdm(train_loader, leave=False):
        loss = model.loss(head, rel, tail)

        opt.zero_grad()
        loss.backward()
        opt.step()



100%|██████████| 20/20 [00:25<00:00,  1.27s/it]
